# WavLM full fine-tuning on Monash Strudel

Dense full fine-tuning of `microsoft/wavlm-large` for 3-class valence on the 854 labelled Gemini Live clips. **CUDA / Strudel only** (not Setonix ROCm).

## Layout on this machine

Everything lives under **your home** project folder (note: `~/Research/...`, not `/Research/...`):

```text
~/Research/tone_bias_2026/
  fine_tuning/          # package + this notebook
  src/config.py
  src/load_config.py
  our_speech_corpus_manifest_split_80_20.csv
  oof_predictions.csv
  our_speech_corpus_final.zip   # unzipped by the Paths cell
  our_speech_corpus_final/      # A_*.wav / B_*.wav
  scratch/wavlm_full_ft/        # caches, fold progress, reports
```

## Request on Strudel

```bash
sbatch --qos=desktopq --partition=desktop --gres=gpu:A40:1 --mem=100G --ntasks=6
```

Use `gpu:T4:1` if A40 is unavailable (then try `BATCH_SIZE = 8` if smoke OOMs).

## Pipeline order

1. Install + GPU check
2. Remap the manifest onto local WAV paths
3. `preflight`
4. `layer-cache` (once)
5. `full-ft-smoke`
6. `full-ft-nested` (all folds on one A40, or one fold at a time)
7. Aggregate (`--full-ft-aggregate-only`) — only needed if you ran per-fold jobs
8. `full-ft-leakage`
9. Inspect reports

Gates A/B need `~/Research/tone_bias_2026/oof_predictions.csv` (partial FT OOF).


## 0. Paths

Run this first. Resolves `~/Research/tone_bias_2026` (or the notebook cwd), unzips audio if needed, and sets `REPO` / `AUDIO` / `MANIFEST` / `PARTIAL_OOF` / `OUT`.


In [ ]:
from pathlib import Path
import os
import sys
import zipfile

# Strudel home project dir — NOT /Research (that is root and permission-denied).
candidates = [
    Path.home() / "Research" / "tone_bias_2026",
    Path.cwd() / "tone_bias_2026",
    Path.cwd(),
]
ROOT = None
for cand in candidates:
    cand = cand.resolve()
    if (cand / "fine_tuning").is_dir() and (cand / "our_speech_corpus_manifest_split_80_20.csv").is_file():
        ROOT = cand
        break
if ROOT is None:
    raise FileNotFoundError(
        "Could not find tone_bias_2026. Looked at:\n  "
        + "\n  ".join(str(c) for c in candidates)
        + "\nOpen the notebook from that folder, or fix Path.home()/Research/tone_bias_2026."
    )

# Unzip audio once if the WAV folder is missing.
AUDIO = ROOT / "our_speech_corpus_final"
ZIP = ROOT / "our_speech_corpus_final.zip"
if not AUDIO.is_dir() and ZIP.is_file():
    print("unzipping", ZIP, "->", ROOT)
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(ROOT)
# Handle accidental double nest from the zip layout.
if AUDIO.is_dir() and not any(AUDIO.glob("*.wav")):
    nested = AUDIO / "our_speech_corpus_final"
    if nested.is_dir() and any(nested.glob("*.wav")):
        AUDIO = nested

REPO = ROOT  # fine_tuning/ and src/ live directly under tone_bias_2026
MANIFEST = ROOT / "our_speech_corpus_manifest_split_80_20.csv"
PARTIAL_OOF = ROOT / "oof_predictions.csv"
OUT = ROOT / "scratch" / "wavlm_full_ft"
OUT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16  # A40
DEVICE = "cuda:0"

os.environ["PYTHONPATH"] = str(REPO) + (
    ":" + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("ROOT", ROOT)
print("REPO", REPO, REPO.is_dir())
print("AUDIO", AUDIO, AUDIO.is_dir(), "n_wav", len(list(AUDIO.glob("*.wav"))) if AUDIO.is_dir() else 0)
print("MANIFEST", MANIFEST, MANIFEST.is_file())
print("PARTIAL_OOF", PARTIAL_OOF, PARTIAL_OOF.is_file())
print("OUT", OUT, OUT.is_dir())
print("fine_tuning", (REPO / "fine_tuning").is_dir(), "config", (REPO / "src" / "config.py").is_file())


## 1. Install dependencies

Use a CUDA PyTorch build that matches the Strudel CUDA module (`nvidia-smi` / module list).


In [ ]:
# Adjust the torch index URL if Strudel's CUDA major version differs.
%pip -q install "torch" "transformers>=4.44" "huggingface_hub" "scikit-learn>=1.5" "soundfile" "scipy" "pandas" "numpy" "accelerate"

import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 2. Remap manifest onto Strudel WAV paths

Manifest paths from your laptop will not match `~/Research/tone_bias_2026/our_speech_corpus_final/`. This writes a labelled-only remapped CSV under `OUT`.


In [ ]:
import pandas as pd
from fine_tuning.data import remap_audio_paths

df = pd.read_csv(MANIFEST)
if "is_labelled" in df.columns:
    df = df[df["is_labelled"].astype(str).str.lower().isin(["true", "1"])].copy()
print("labelled rows", len(df))

remapped = remap_audio_paths(df, AUDIO, strict=True)
MANIFEST_LOCAL = OUT / "manifest_remapped.csv"
remapped.to_csv(MANIFEST_LOCAL, index=False)
print("wrote", MANIFEST_LOCAL)
print(remapped[["clip_id", "audio_path"]].head(3).to_string(index=False))


## 3. Preflight

Checks WavLM load, freeze plan, and suffix parity on CUDA.


In [ ]:
!python "{REPO}/fine_tuning/train_wavlm_partial.py" \
  --stage preflight \
  --manifest "{MANIFEST_LOCAL}" \
  --corpora-root "{AUDIO}" \
  --output-dir "{OUT}" \
  --device "{DEVICE}"


## 4. Layer cache

Run once. Cached frame pools are reused by nested CV and the matched frozen branch. Writes under `OUT/cache/`.


In [ ]:
!python "{REPO}/fine_tuning/train_wavlm_partial.py" \
  --stage layer-cache \
  --manifest "{MANIFEST_LOCAL}" \
  --corpora-root "{AUDIO}" \
  --output-dir "{OUT}" \
  --device "{DEVICE}"


## 5. Full-FT smoke

fp32 forward + backward on the longest clips (CNN frozen, all 24 transformers trainable). Confirms `BATCH_SIZE` fits on this GPU before nested CV.


In [ ]:
!python "{REPO}/fine_tuning/train_wavlm_partial.py" \
  --stage full-ft-smoke \
  --manifest "{MANIFEST_LOCAL}" \
  --corpora-root "{AUDIO}" \
  --output-dir "{OUT}" \
  --device "{DEVICE}" \
  --full-ft-batch-size {BATCH_SIZE} \
  --full-ft-smoke-clips {BATCH_SIZE}


## 6. Nested CV — one outer fold

Set `FOLD` to `0`…`4` and re-run if you want to checkpoint per fold. Prefer the next cell (all folds) on a long A40 session.

Primary arm: uniform mean-pool over 25 hidden states + MLP. Uses `PARTIAL_OOF` for gates A/B.


In [ ]:
FOLD = 0  # change to 1..4 for the other outer folds

!python "{REPO}/fine_tuning/train_wavlm_partial.py" \
  --stage full-ft-nested \
  --manifest "{MANIFEST_LOCAL}" \
  --corpora-root "{AUDIO}" \
  --output-dir "{OUT}" \
  --partial-oof-csv "{PARTIAL_OOF}" \
  --full-ft-outer-fold {FOLD} \
  --full-ft-batch-size {BATCH_SIZE} \
  --device "{DEVICE}"


### Optional: run all five folds in this session

Omits `--full-ft-outer-fold`, so folds `0..4` run sequentially on the single A40 and then aggregate automatically. Skip the dedicated aggregate cell if you use this.


In [ ]:
# All outer folds sequentially on one GPU (omit --full-ft-outer-fold).
!python "{REPO}/fine_tuning/train_wavlm_partial.py" \
  --stage full-ft-nested \
  --manifest "{MANIFEST_LOCAL}" \
  --corpora-root "{AUDIO}" \
  --output-dir "{OUT}" \
  --partial-oof-csv "{PARTIAL_OOF}" \
  --full-ft-batch-size {BATCH_SIZE} \
  --device "{DEVICE}"


## 7. Aggregate completed folds

CPU-safe. Only needed if you ran **per-fold** jobs (section 6 with `FOLD=`). Reads `OUT/full_ft_progress/fold*.json` and writes nested-CV / paired reports under `OUT/reports/`.


In [ ]:
!python "{REPO}/fine_tuning/train_wavlm_partial.py" \
  --stage full-ft-nested \
  --manifest "{MANIFEST_LOCAL}" \
  --corpora-root "{AUDIO}" \
  --output-dir "{OUT}" \
  --partial-oof-csv "{PARTIAL_OOF}" \
  --full-ft-aggregate-only \
  --device cpu


## 8. Leakage check

Mandatory grouped-vs-random refits at the locked full-FT config. Writes `OUT/reports/full_ft_leakage.json`.


In [ ]:
!python "{REPO}/fine_tuning/train_wavlm_partial.py" \
  --stage full-ft-leakage \
  --manifest "{MANIFEST_LOCAL}" \
  --corpora-root "{AUDIO}" \
  --output-dir "{OUT}" \
  --partial-oof-csv "{PARTIAL_OOF}" \
  --full-ft-batch-size {BATCH_SIZE} \
  --device "{DEVICE}"


## 9. Inspect reports

Looks under `OUT/reports/` (i.e. `~/Research/tone_bias_2026/scratch/wavlm_full_ft/reports/`).


In [ ]:
import json

reports = OUT / "reports"
if not reports.is_dir():
    print("no reports dir yet:", reports)
else:
    for name in sorted(p.name for p in reports.glob("*.json")):
        print(name)

    # stages.py writes full_ft_nested_cv.json
    for candidate in ("full_ft_nested_cv.json", "full_ft_nested.json"):
        nested = reports / candidate
        if nested.is_file():
            payload = json.loads(nested.read_text())
            print("\nloaded", nested.name)
            print("keys", sorted(payload.keys())[:20])
            for key in ("headline", "gates", "protocol", "primary_arm"):
                if key in payload:
                    print(f"\n=== {key} ===")
                    print(json.dumps(payload[key], indent=2)[:2000])
            break
    else:
        print("nested CV JSON not written yet — finish folds + aggregate")


## Files expected under `~/Research/tone_bias_2026/`

| Path | Why |
|---|---|
| `fine_tuning/` (entire package + this notebook) | CLI / stages / model |
| `src/config.py`, `src/load_config.py` | bootstrap |
| `our_speech_corpus_manifest_split_80_20.csv` | labels + questions |
| `oof_predictions.csv` | partial FT OOF for gates A/B |
| `our_speech_corpus_final.zip` → unzipped `our_speech_corpus_final/*.wav` | audio |

**Downloaded once on the node:** Hugging Face `microsoft/wavlm-large`.

**Written under `OUT`:** `cache/`, `full_ft_progress/fold{0-4}.json`, `reports/full_ft_nested_cv.json`, paired / leakage JSON.

## Notes

- Path is under your **home** (`Path.home()/Research/...`). Absolute `/Research` is not writable.
- One GPU (`A40:1`); folds run sequentially in this notebook.
- `--mem=100G` is host RAM, not VRAM — VRAM is controlled with `--full-ft-batch-size`.
- Keep `--device cuda:0`.
